In [ ]:
1+1

### RAG with MongoDB - DATA ingestion, Retrieval And Generation

## Data Ingestion

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Load embedding model (FREE)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}  # avoid GPU errors
)

# Function to get embedding
def get_embedding(text):
    return embeddings.embed_query(text)

# Test
print(get_embedding("AI Technology"))
len(get_embedding("AI Technology"))

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("https://investors.mongodb.com/node/12236/pdf")
data = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=20)
documents = text_splitter.split_documents(data)

In [ ]:
documents

### Prepare doc for insertion


In [ ]:
docs_to_insert = [{
    "text": doc.page_content,
    "embedding": get_embedding(doc.page_content)
} for doc in documents]

In [ ]:
docs_to_insert

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

client = MongoClient(os.getenv("MONGO_URI"))

In [ ]:
load_dotenv()

In [ ]:
print(os.getenv("MONGO_URI"))

In [ ]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv("../.env")  # adjust if needed

uri = os.getenv("MONGO_URI")
print(uri)  # DEBUG

client = MongoClient(uri)

collection = client["sample_mflix"]["RAGpdf"]

collection.delete_many({})  # Clear existing data

result = collection.insert_many(docs_to_insert)

print(result.inserted_ids)

In [ ]:
from pymongo.operations import SearchIndexModel
import time

index_name = "vector_index"

search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {
                "type": "vector",
                "numDimensions": 384,  # ✅ FIXED
                "path": "embedding",
                "similarity": "cosine"
            }
        ]
    },
    name=index_name,
    type="vectorSearch"
)

collection.create_search_index(model=search_index_model)

print("Polling to check if the index is ready...")

while True:
    indices = list(collection.list_search_indexes(index_name))
    if len(indices) and indices[0].get("queryable") is True:
        break
    time.sleep(5)

print(index_name + " is ready for querying.")

In [ ]:
query_embedding = get_embedding("AI Technology") 
query_embedding

In [ ]:
query_embedding = get_embedding("AI Technology")
collection.test.aggregate([
  {
    "$vectorSearch": {
      "index": "vector_index",
      "path": "embedding",
      "queryVector":query_embedding,
      "numCandidates":384 ,
      "limit": 5
    }
  }
])

In [ ]:
# Define a function to run vector search queries
def get_query_results(query):
  """Gets results from a vector search query."""

  query_embedding = get_embedding(query)
  print(query_embedding)
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index",
              "queryVector": query_embedding,
              "path": "embedding",
              "numCandidates":384,
              "limit": 5
            }
      }, {
            "$project": {
              "_id": 0,
              "text": 1
         }
      }
  ]

  results = collection.aggregate(pipeline)
  print(results)

  array_of_results = []
  for doc in results:
      array_of_results.append(doc)
  return array_of_results



In [ ]:
# Test the function with a sample query
get_query_results("mongodb vector search")

In [ ]:
from transformers import pipeline

# Load local model
pipe = pipeline(
    "text-generation",
    model="google/flan-t5-small",  # fast + works on CPU
    max_new_tokens=200,
    device=-1  # force CPU
)

In [ ]:
# Query
query = "What are MongoDB's latest AI announcements?"

# Retrieve documents
context_docs = get_query_results(query)

# Convert to string
context_string = " ".join([doc["text"] for doc in context_docs])

# Prompt
prompt = f"""
You are an AI assistant.

Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't know".

Context:
{context_string}

Question:
{query}

Answer:
"""

# Generate answer (HuggingFace)
response = pipe(prompt)

# Extract text
answer = response[0]["generated_text"]

print(answer)